# Foundation Model Probes
**SUPERSEDED VALIDATION FRAMEWORK.** Maintained EEGPT has been tested negatively and the exact official route remains blocked. Do not use this old framework as a rerun path; see `AGENTS.md` section 2e. Execution fails closed.

# 1. Setup

In [ ]:
from __future__ import annotations
raise RuntimeError('CLOSED superseded foundation-probe framework: see AGENTS.md section 2e.')
import builtins, hashlib, json, os, platform, random, sys, time
from datetime import datetime
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.stats import wilcoxon
from sklearn.metrics import confusion_matrix
from modern_mi_common import *
print(f"Python: {sys.version.split()[0]} | Platform: {platform.platform()} | CWD: {Path.cwd()}")

# 2. Configuration
## 2.1 Official Input Contracts
EEGPT: 58 channels, 256 Hz, 4 s. CBraMod: 200 Hz patch input.
## 2.2 CONFIG

In [ ]:
WORKING_DIR = Path.cwd().resolve().parent.parent
CONFIG = {
    # Paths / run identity
    "artifact_dir": str(WORKING_DIR / "artifacts" / "liu2024-foundation-model-probes"), "source_extract_dir": str(WORKING_DIR / "liu2024_data" / "liu2024_figshare" / "sourcedata"), "experiment_name": "foundation_eegpt_validate", "config_note": "Fail-closed official asset/API validation.",
    # Foundation provenance (all required; no downloads)
    "foundation_model": "EEGPT", "validate_only": True, "repository_path": None, "repository_revision": None, "checkpoint_path": None, "checkpoint_sha256": None, "module_import": None, "model_factory": None, "feature_method": None,
    # Explicit input/mapping contract
    "expected_channels": 58, "target_sfreq": 256, "window_seconds": 4.0, "channel_mapping": None, "interpolation_matrix_path": None, "patch_seconds": None,
    # Probe / evaluation / reproducibility
    "probe_models": ["shrinkage_lda","logistic"], "adapter_mode": "none", "adapter_rank": 4, "subjects_to_use": None, "cv_folds": 5, "cv_random_state": 2026, "seed": 2026, "set_seed": True
}

## 2.3 Artifact Creation and Logging Init

In [ ]:
def create_run_id():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    config_hash = hashlib.md5(json.dumps(CONFIG, sort_keys=True, default=str).encode()).hexdigest()[:8]
    return f"{timestamp}_{config_hash}"
RUN_ID = create_run_id(); ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=False)
LOG_PATH = ARTIFACT_DIR / "run.log"; _LOG_FILE_HANDLE = open(LOG_PATH, "a", buffering=1, encoding="utf-8", errors="replace")
def _safe_write_text(stream, text):
    try: stream.write(text)
    except UnicodeEncodeError:
        enc = getattr(stream, "encoding", None) or "utf-8"; stream.write(text.encode(enc, errors="replace").decode(enc, errors="replace"))
def _timestamped_print(*args, **kwargs):
    sep=kwargs.pop("sep"," "); end=kwargs.pop("end","\n"); flush=kwargs.pop("flush",False); file=kwargs.pop("file",None)
    message=sep.join(str(a) for a in args); target=sys.stdout if file is None else file; stamped=f"[{datetime.now():%Y-%m-%d %H:%M:%S}] {message}"
    _safe_write_text(target, stamped+end); _safe_write_text(_LOG_FILE_HANDLE, stamped+end)
    if flush: target.flush(); _LOG_FILE_HANDLE.flush()
builtins.print = _timestamped_print
config_path=ARTIFACT_DIR/"config.json"; config_path.write_text(json.dumps(CONFIG,indent=2),encoding="utf-8")
print(f"Run ID:     {RUN_ID}"); print(f"Artifacts:  {ARTIFACT_DIR}"); print(f"Config:     {config_path}")

## 2.4 Reproducibility

In [ ]:
def resolve_device():
    if torch.backends.mps.is_available() and torch.backends.mps.is_built(): return torch.device("mps")
    if torch.cuda.is_available(): return torch.device("cuda")
    return torch.device("cpu")
DEVICE=resolve_device(); print(f"Using device: {DEVICE}")
def seed_everything(seed):
    os.environ["PYTHONHASHSEED"]=str(seed); random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed); torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True
    torch.use_deterministic_algorithms(True,warn_only=True)
BASE_SEED=int(CONFIG["seed"])
if CONFIG["set_seed"]: seed_everything(BASE_SEED); print(f"Seed initialized: {BASE_SEED}")

# 3. Load and Prepare Data
## 3.1 Asset and API Validation
## 3.2 Explicit Channel Mapping
## 3.3 Fold-Local Probe Transforms
## 3.4 Locate Assets

# 4. Model
## 4.1 Frozen Feature Model and Probe
## 4.2 Optional Token-Feature Adapter
Adapters are permitted only after the configured feature API validates.

# 5. Training
## 5.1 Shrinkage LDA and Logistic Probes
## 5.2 Within-Subject Cross-Validation Runner
## 5.3 Validate or Run

In [ ]:
print("Full CONFIG banner:\n"+json.dumps(CONFIG,indent=2))
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
    return h.hexdigest()
def validate_assets(cfg):
    required=["repository_path","repository_revision","checkpoint_path","checkpoint_sha256","module_import","model_factory","feature_method"]
    missing=[k for k in required if not cfg.get(k)]
    if missing: raise FileNotFoundError("Missing required official asset/provenance fields: "+", ".join(missing)+". Supply local pinned repo/checkpoint; this notebook never downloads.")
    repo=Path(cfg["repository_path"]); ckpt=Path(cfg["checkpoint_path"])
    if not repo.is_dir() or not ckpt.is_file(): raise FileNotFoundError(f"Repository/checkpoint absent: {repo}, {ckpt}")
    actual=sha256_file(ckpt)
    if actual.lower()!=cfg["checkpoint_sha256"].lower(): raise RuntimeError(f"Checkpoint SHA256 mismatch: expected {cfg['checkpoint_sha256']}, got {actual}")
    if cfg["foundation_model"]=="EEGPT" and (cfg["expected_channels"],cfg["target_sfreq"],cfg["window_seconds"])!=(58,256,4.0): raise ValueError("EEGPT official expected input is 58ch/256Hz/4s")
    if cfg["foundation_model"]=="CBraMod" and (cfg["target_sfreq"]!=200 or not cfg["patch_seconds"]): raise ValueError("CBraMod requires explicit 200Hz patch configuration")
    if not cfg.get("channel_mapping") and not cfg.get("interpolation_matrix_path"): raise ValueError("Explicit Liu-to-official channel mapping/interpolation is required")
    sys.path.insert(0,str(repo)); import importlib; module=importlib.import_module(cfg["module_import"]); factory=getattr(module,cfg["model_factory"]); model=factory(); load=torch.load(ckpt,map_location="cpu",weights_only=False); model.load_state_dict(load.get("state_dict",load),strict=True); model.eval()
    if not hasattr(model,cfg["feature_method"]): raise AttributeError(f"Model does not expose configured feature method {cfg['feature_method']}")
    return {"repository_path":str(repo.resolve()),"repository_revision":cfg["repository_revision"],"checkpoint_path":str(ckpt.resolve()),"checkpoint_sha256":actual,"module_import":cfg["module_import"],"model_factory":cfg["model_factory"],"feature_method":cfg["feature_method"]}
PROVENANCE=validate_assets(CONFIG)
if not CONFIG["validate_only"]: raise NotImplementedError("Full extraction is intentionally gated until an official API/checkpoint validates locally; add only against that pinned API.")

# 6. Results
## 6.1 Aggregate Metrics
Validation-only runs contain provenance rather than performance claims.
## 6.2 Performance Visualizations
Not applicable to validation-only mode.
## 6.3 Experiment Summary

In [ ]:
print(json.dumps(PROVENANCE,indent=2))

## 6.4 Provenance Diagnostics
Pinned revision, checkpoint digest, import/factory, and feature method are saved.

## 6.5 Save Artifacts

In [ ]:
(ARTIFACT_DIR/"foundation_provenance.json").write_text(json.dumps(PROVENANCE,indent=2),encoding="utf-8"); FOLD_RESULTS=[]; SUBJECT_ROWS=[]; GLOBAL_METRICS={"validate_only":True,"status":"validated"}; SUBJECTS=[]; CH_NAMES=[]; subject_inventory_path=ARTIFACT_DIR/"subject_inventory.csv"; pd.DataFrame(columns=["subject_id"]).to_csv(subject_inventory_path,index=False)
cv_results_path=ARTIFACT_DIR/"cv_results.json"; cv_results_path.write_text(json.dumps(FOLD_RESULTS,indent=2),encoding="utf-8")
subject_metrics_path=ARTIFACT_DIR/"subject_metrics.json"; subject_metrics_path.write_text(json.dumps(SUBJECT_ROWS,indent=2),encoding="utf-8")
global_metrics_path=ARTIFACT_DIR/"global_metrics.json"; global_metrics_path.write_text(json.dumps(GLOBAL_METRICS,indent=2),encoding="utf-8")
pd.DataFrame(FOLD_RESULTS).to_csv(ARTIFACT_DIR/"fold_results.csv",index=False); pd.DataFrame(SUBJECT_ROWS).to_csv(ARTIFACT_DIR/"subject_results.csv",index=False)
run_metadata={"run_id":RUN_ID,"artifact_dir":str(ARTIFACT_DIR),"experiment_name":CONFIG["experiment_name"],"config_note":CONFIG["config_note"],"subjects":SUBJECTS,"channel_names":CH_NAMES,"model_name":CONFIG.get("model_name"),"seed":BASE_SEED,"global_metrics":GLOBAL_METRICS,"artifacts":{p.name:str(p) for p in ARTIFACT_DIR.iterdir()}}
run_metadata_path=ARTIFACT_DIR/"run_metadata.json"; run_metadata_path.write_text(json.dumps(run_metadata,indent=2),encoding="utf-8")
print(f"CV results saved to:      {cv_results_path}"); print(f"Subject metrics saved to: {subject_metrics_path}"); print(f"Global metrics saved to:  {global_metrics_path}"); print(f"Run metadata saved to:    {run_metadata_path}"); print(f"\nAll artifacts in: {ARTIFACT_DIR}")
try: _LOG_FILE_HANDLE.close()
except Exception: pass